# Phonon computations

This is a quick sketch how to run a simple phonon calculation using DFTK.

> **Preliminary implementation**
>
> Practical phonon computations have only seen rudimentary testing as of now.
> As of now we do not yet recommend relying on this feature for production
> calculations. We appreciate any issues, bug reports or PRs.

First we run an SCF calculation.

In [1]:
using AtomsBuilder
using DFTK
using PseudoPotentialData

pseudopotentials = PseudoFamily("cp2k.nc.sr.lda.v0_1.semicore.gth")
model  = model_DFT(bulk(:Si); pseudopotentials, functionals=LDA())
basis  = PlaneWaveBasis(model; Ecut=10, kgrid=[4, 4, 4])
scfres = self_consistent_field(basis, tol=1e-8);
nothing  # hide

n     Energy            log10(ΔE)   log10(Δρ)   Diag   Δtime
---   ---------------   ---------   ---------   ----   ------
  1   -7.916382757323                   -0.69    4.8    161ms
  2   -7.921195996622       -2.32       -1.52    1.0   56.1ms
  3   -7.921406177390       -3.68       -2.49    1.5   63.1ms
  4   -7.921440965731       -4.46       -2.84    2.6   76.6ms
  5   -7.921441676294       -6.15       -3.05    1.1   56.3ms
  6   -7.921442008280       -6.48       -4.54    1.1   56.5ms
  7   -7.921442021898       -7.87       -4.67    2.9    109ms
  8   -7.921442022134       -9.63       -5.38    1.0   56.2ms
  9   -7.921442022127   +  -11.18       -4.88    1.9   65.4ms
 10   -7.921442022144      -10.79       -5.98    1.0   57.3ms
 11   -7.921442022144      -12.30       -6.91    1.4   59.8ms
 12   -7.921442022144      -13.55       -6.63    2.5   82.1ms
 13   -7.921442022144      -14.05       -7.76    1.0   55.6ms
 14   -7.921442022144   +  -14.75       -8.46    1.8    116ms


Next we compute the phonon modes at the q-point `[1/4, 1/4, 1/4]`.

In [2]:
scfres = DFTK.unfold_bz(scfres)
phret_q0 = @time DFTK.phonon_modes(scfres; q=[0.25, 0.25, 0.25]);
nothing  # hide

Iter  Restart  Krydim  log10(res)  avg(CG)  Δtime   Comment
----  -------  ------  ----------  -------  ------  ---------------
                                      73.0   2.44s  Non-interacting
   1        0       1        0.31     52.4   9.42s  
   2        0       2       -0.52     47.1   1.67s  
   3        0       3       -2.40     41.7   1.46s  
   4        0       4       -4.46     29.7   1.16s  
   5        0       5       -6.46     17.4   808ms  
   6        0       6       -8.91      4.0   447ms  
   7        1       1       -7.84     55.6   2.26s  Restart
   8        1       2       -8.85      5.5   517ms  
                                      72.8   2.93s  Final orbitals
Iter  Restart  Krydim  log10(res)  avg(CG)  Δtime   Comment
----  -------  ------  ----------  -------  ------  ---------------
                                      72.9   2.03s  Non-interacting
   1        0       1        0.31     52.4   1.87s  
   2        0       2       -0.52     47.1   1.61s  
   3

These are the final phonon frequencies:

In [3]:
phret_q0.frequencies

6-element Vector{Float64}:
 -0.003498155350424302
 -0.002977451151364397
 -0.0029774511510520596
  0.004302230637630997
  0.004302230637730469
  0.004353201767506925